# VanDerPol — Stage 1 — SINDy System Identification

Fit and validate a sparse SINDy dynamics model.


## 1. Set up the system and load the stage config

Load dependencies and configuration.


In [ ]:
"""Boilerplate: make the in-repo `sdpc` package importable and resolve this system."""
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

_p = Path.cwd()
while not (_p / "src" / "sdpc").exists():
    _p = _p.parent
sys.path.insert(0, str(_p / "src"))

import torch
from sdpc.config import load_config
from sdpc.registry import make_system

SYSTEM = "vanderpol_relative_degree_one"
device = torch.device("cpu")
system = make_system(SYSTEM, device=device)
CONFIGS = Path.cwd().parent / "configs"
RESULTS = Path.cwd().parent / "results"

print(f"System        : {SYSTEM}")
print(f"State dim nx  : {system.nx}")
print(f"Control dim nu: {system.nu}")
print(f"Sample time ts: {system.ts}")
print(f"Input bounds  : [{system.umin}, {system.umax}]")
print(f"State bounds  : [{system.xmin}, {system.xmax}]")
print(f"Discrete model: {system.is_discrete}")


In [ ]:
cfg = load_config(CONFIGS / 'sysid.yaml')
cfg['epochs'] = 3000  # reduced for interactive use; remove this line for the full run

print('Resolved system-ID config:')
for k, v in cfg.items():
    if not k.startswith('_'):
        print(f'  {k}: {v}')

## 2. Build the candidate feature library and the SINDy model

Build the candidate sparse model.


In [ ]:
from sdpc.sindy import CompiledFunctionLibrary, SINDyVectorized

lib_cfg = system.sindy_library_cfg()
print('Library config:', lib_cfg)

lib = CompiledFunctionLibrary(**lib_cfg)
sindy = SINDyVectorized(library=lib, n_out=system.nx, device=device)

print(f'\nCandidate library has {lib.shape[0]} terms:')
print(lib.function_names)

## 3. Generate system-identification data

Prepare the required training data.


In [ ]:
train_loader, dev_loader, test_data = system.make_sysid_data(cfg, device)

batch = next(iter(train_loader))
print('Train batch keys :', list(batch.keys()))
print('X shape (batch, horizon, nx):', tuple(batch['X'].shape))
print('u shape (batch, horizon, nu):', tuple(batch['u'].shape))
print('Test rollout length:', test_data['X'].shape[1])

## 4. Train the sparse dynamics model (Algorithm 1)

Fit and prune the sparse model.


In [ ]:
from sdpc.training import train_sysid
from sdpc.io import CustomLogger

logger = CustomLogger(args=None, savedir=str(RESULTS / 'logs' / 'sysid_nb'),
                      verbosity=max(1, cfg['epochs'] // 10),
                      stdout=['train_loss', 'dev_loss'])
train_sysid(system, sindy, train_loader, dev_loader, cfg, device, logger=logger)

## 5. Inspect the identified model

Inspect the learned dynamics.


In [ ]:
print('Identified sparse dynamics:')
sindy.pretty_print()
print()
print('Active terms per state:', [len(idx) for idx in sindy.active_idx])

## 6. Validate: predicted vs. true trajectory on held-out test data

Visualize the closed-loop response.


In [ ]:
import matplotlib.pyplot as plt

step = system.discrete_step(sindy)
X_true = test_data['X'][0]   # (T, nx) recorded true trajectory
U_test = test_data['u'][0]   # (T, nu) recorded inputs

x = X_true[:1]
pred = [x]
with torch.no_grad():
    for k in range(len(U_test) - 1):
        x = step(x, U_test[k:k+1])
        pred.append(x)
pred = torch.cat(pred, dim=0).detach().numpy()
true = X_true.detach().numpy()

mse = ((pred - true) ** 2).mean()
print(f'Open-loop prediction MSE over the test rollout: {mse:.3e}')

fig, ax = plt.subplots(system.nx, 1, figsize=(9, 2.5 * system.nx), sharex=True)
ax = ax if system.nx > 1 else [ax]
for i in range(system.nx):
    ax[i].plot(true[:, i], 'c', lw=2.5, label='true')
    ax[i].plot(pred[:, i], 'm--', lw=2.0, label='predicted (SINDy)')
    ax[i].set_ylabel(f'x{i}'); ax[i].grid(alpha=.3); ax[i].legend()
ax[-1].set_xlabel('time step')
fig.suptitle(f'{SYSTEM}: SINDy open-loop prediction vs. ground truth')
plt.tight_layout(); plt.show()

## 7. Save the model (optional)

To save a legacy run directly from this notebook, uncomment the cell below.

In [ ]:
# from sdpc.sindy import save_model
# from sdpc.io import new_run_dir, snapshot_config
# run_dir = new_run_dir(RESULTS / 'models' / 'dynamics')
# save_model(sindy, run_dir / 'saved_models' / 'sindy.pt')
# snapshot_config(run_dir, cfg)
# print('Saved to', run_dir)